# Solutions · 06 · Differentiation of data: identifying a cooling law

Worked solutions to the exercises of [notebook 06](../notebooks/06_differentiation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import differentiate, fitting, smoothing
T_inf, dT0, C, n_true = 20.0, 480.0, 5.0e-4, 1.25
t = np.r_[np.arange(0, 60, 5), np.arange(60, 300, 20), np.arange(300, 1801, 100)].astype(float)
dT = (dT0**-0.25 + C*t/4)**-4
T = T_inf + dT
T_noisy = T + np.random.default_rng(3).normal(0, 0.5, T.size)

def fit_n(temps, derivs):
    ok = (derivs < 0) & (temps > T_inf)
    f = fitting.linear_lstsq(np.column_stack([np.ones(ok.sum()), np.log(temps[ok] - T_inf)]), np.log(-derivs[ok]))
    return f.coef[1], f.conf_int()[1]

## Exercise 1

Refit using only the first 10 minutes of data. How does the confidence interval of $n$ change?

In [2]:
for label, temps in (("noise-free", T), ("noisy", T_noisy)):
    for tmax, part in ((1800, "all data"), (600, "first 10 min")):
        m = t <= tmax
        n_est, ci = fit_n(temps[m], differentiate.gradient(t[m], temps[m]))
        print(f"{label:<10} {part:<12} ({m.sum():2d} points): n = {n_est:.3f}, 95 % CI [{ci[0]:.3f}, {ci[1]:.3f}]")

noise-free all data     (40 points): n = 1.248, 95 % CI [1.247, 1.250]
noise-free first 10 min (28 points): n = 1.253, 95 % CI [1.246, 1.259]
noisy      all data     (40 points): n = 1.249, 95 % CI [1.211, 1.288]
noisy      first 10 min (28 points): n = 1.290, 95 % CI [1.137, 1.443]


For the noisy data the confidence interval **widens** when only the first ten minutes are used: fewer
points, and - more importantly - a narrower range of temperature differences, so the slope of
$\ln(-dT/dt)$ against $\ln(T - T_\infty)$ is less well determined. A wide range of the independent variable
is what makes a slope precise. (For noise-free data both intervals are tiny: the only error left is the
small truncation error of the differences.)

## Exercise 2

Use `differentiate.forward` differences instead (first order). How large is the bias in $n$
   for the noise-free data?

In [3]:
fwd = np.r_[np.diff(T) / np.diff(t), np.nan]            # forward differences (last point has none)
n_fwd, _ = fit_n(T[:-1], fwd[:-1])
n_cen, _ = fit_n(T, differentiate.gradient(t, T))
print(f"forward differences: n = {n_fwd:.4f};  second-order formulas: n = {n_cen:.4f};  true 1.25")

forward differences: n = 1.2828;  second-order formulas: n = 1.2483;  true 1.25


The forward difference estimates the slope at the *middle* of each interval but assigns it to the start,
where the true slope is steeper. The error is largest where the curve bends most - early on - and it biases
the fitted exponent. The second-order formulas remove almost all of this bias.

## Exercise 3

Estimate $n$ from the noisy data by smoothing first with `engmath.smoothing.lowess`. Does the
   estimate improve?

In [4]:
for frac in (0.15, 0.25, 0.4):
    T_smooth = smoothing.lowess(t, T_noisy, frac=frac, degree=2)
    n_s, ci = fit_n(T_smooth, differentiate.gradient(t, T_smooth))
    print(f"LOWESS (frac {frac}): n = {n_s:.3f}, 95 % CI [{ci[0]:.3f}, {ci[1]:.3f}]")
n_raw, ci = fit_n(T_noisy, differentiate.gradient(t, T_noisy))
print(f"no smoothing:       n = {n_raw:.3f}, 95 % CI [{ci[0]:.3f}, {ci[1]:.3f}]")

LOWESS (frac 0.15): n = 1.231, 95 % CI [1.210, 1.253]
LOWESS (frac 0.25): n = 1.248, 95 % CI [1.236, 1.260]
LOWESS (frac 0.4): n = 1.369, 95 % CI [1.277, 1.462]
no smoothing:       n = 1.249, 95 % CI [1.211, 1.288]


Moderate smoothing (frac 0.25) narrows the confidence interval and keeps the estimate close to 1.25. Too
much smoothing (frac 0.4) distorts the curve: the estimate moves well away from 1.25 and its interval
**excludes the true value**. The interval is computed as if the smoothed points were independent
measurements, which they are not, and it ignores the bias that smoothing introduces - so it can be badly
misleading. Smoothing trades noise for bias. Fitting the integrated model directly
(notebook 09, exercise 3) avoids differentiation altogether.

## Exercise 4

**Implement it yourself:** using the same parabola, derive the one-sided formula for the *first* point (differentiate at $t_0$ instead of $t_1$). Implement it and check it against `differentiate.gradient` at $t = 0$.

In [5]:
import sympy as sp
s, h1, h2 = sp.symbols("s h1 h2", positive=True)
f0, f1, f2 = sp.symbols("f0 f1 f2")
parabola = sp.interpolate([(0, f0), (h1, f1), (h1 + h2, f2)], s)
start = sp.expand(sp.diff(parabola, s).subs(s, 0))
for fi in (f0, f1, f2):
    print(fi, ":", sp.factor(start.coeff(fi)))
formula = sp.lambdify((f0, f1, f2, h1, h2), start)
print("formula:", formula(T[0], T[1], T[2], t[1] - t[0], t[2] - t[1]), " library:", differentiate.gradient(t, T)[0])

f0 : -(2*h1 + h2)/(h1*(h1 + h2))
f1 : (h1 + h2)/(h1*h2)
f2 : -h1/(h2*(h1 + h2))
formula: -1.1232720342838363  library: -1.1232720342838434


Differentiating the same parabola at its first point gives
$f'(t_0) \approx -\frac{2h_1 + h_2}{h_1(h_1 + h_2)}f_0 + \frac{h_1 + h_2}{h_1h_2}f_1 - \frac{h_1}{h_2(h_1 + h_2)}f_2$, which is exactly what
`differentiate.gradient` uses at the first point. It is second order too, but with a larger error constant
than the central formula, because it extrapolates rather than interpolates.